# Medical Diagnostic & Explainable AI Pipeline for Diabetic Retinopathy
### End-to-End Clinical Transfer Learning (ResNet50), Ben Graham Preprocessing, and Grad-CAM

This notebook is fully self-contained and ready to execute on **Google Colab** (with free T4 GPU) or **Kaggle**.

#### Clinical Background: International Clinical Diabetic Retinopathy (ICDR) Scale
- **Grade 0 (No DR)**: Normal fundus, no microvascular lesions.
- **Grade 1 (Mild NPDR)**: Microaneurysms only.
- **Grade 2 (Moderate NPDR)**: Microaneurysms, dot/blot hemorrhages, hard exudates, cotton-wool spots.
- **Grade 3 (Severe NPDR)**: 4-2-1 rule (>20 hemorrhages in 4 quadrants, venous beading in 2+, IRMA in 1+).
- **Grade 4 (Proliferative DR)**: Neovascularization (NVD/NVE), vitreous/preretinal hemorrhage.

## 1. Environment & Hardware Verification

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. Dependencies Installation
Install `opencv-python`, `scikit-learn`, and plotting libraries if running on a fresh Colab environment.

In [ ]:
!pip install -q opencv-python matplotlib seaborn pandas scikit-learn pillow

## 3. Data Acquisition (APTOS 2019 / EyePACS Dataset)

### On Kaggle:
If you create a Kaggle notebook, simply click **+ Add Data** on the right sidebar and search for `aptos2019-blindness-detection`.

### On Google Colab:
Upload your `kaggle.json` API token (from your Kaggle Profile -> Settings -> Create New Token) to Colab:

In [ ]:
import os

# If running on Colab with Kaggle API:
if os.path.exists('kaggle.json'):
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    !kaggle competitions download -c aptos2019-blindness-detection
    !unzip -q aptos2019-blindness-detection.zip -d aptos_data
    DATA_DIR = 'aptos_data/train_images'
    CSV_PATH = 'aptos_data/train.csv'
elif os.path.exists('/kaggle/input/aptos2019-blindness-detection'):
    # Kaggle environment path
    DATA_DIR = '/kaggle/input/aptos2019-blindness-detection/train_images'
    CSV_PATH = '/kaggle/input/aptos2019-blindness-detection/train.csv'
else:
    print("Note: Dataset directory not found automatically. Please provide DATA_DIR and CSV_PATH.")

## 4. Medical Preprocessing: Ben Graham's Retinal Normalization

In fundus imaging, images captured from different clinic camera models have inconsistent pupil lighting, uneven illumination, and large black borders.
Ben Graham's award-winning preprocessing method:
1. **Crop black border**: Isolate the spherical retina contour.
2. **Resize**: Rescale to target dimension ($224 \times 224$ or $299 \times 299$).
3. **Local lighting subtraction**: `4 * image - 4 * GaussianBlur(image) + 128` to equalize brightness.
4. **Circular mask**: Mask out outer sensor border artifacts.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def crop_retina_border(img_rgb, tol=12):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    mask = gray > tol
    if not np.any(mask):
        return img_rgb
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    ymin, ymax = np.where(rows)[0][[0, -1]]
    xmin, xmax = np.where(cols)[0][[0, -1]]
    return img_rgb[ymin:ymax+1, xmin:xmax+1]

def apply_circular_mask(img):
    h, w = img.shape[:2]
    center = (int(w / 2), int(h / 2))
    radius = int(min(h, w) * 0.48)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(mask, center, radius, (255, 255, 255), -1)
    return cv2.bitwise_and(img, img, mask=mask)

def ben_graham_preprocessing(img_rgb, target_size=224, sigma_x=10):
    cropped = crop_retina_border(img_rgb)
    resized = cv2.resize(cropped, (target_size, target_size), interpolation=cv2.INTER_AREA)
    blurred = cv2.GaussianBlur(resized, (0, 0), sigmaX=sigma_x)
    equalized = cv2.addWeighted(resized, 4, blurred, -4, 128)
    return apply_circular_mask(equalized)

## 5. Dataset Loader & Data Augmentation Pipeline

In [ ]:
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

class APTOSRetinaDataset(Dataset):
    def __init__(self, df, img_dir, img_size=224, is_training=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.img_size = img_size
        
        if is_training:
            self.transform = transforms.Compose([
                transforms.ToPILImage(),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.5),
                transforms.RandomRotation(degrees=30),
                transforms.ColorJitter(brightness=0.1, contrast=0.1),
                transforms.ToTensor(),
                transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
            ])
        else:
            self.transform = transforms.Compose([
                transforms.ToPILImage(),
                transforms.ToTensor(),
                transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
            ])
            
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = str(row['id_code'])
        img_path = os.path.join(self.img_dir, f"{img_name}.png")
        if not os.path.exists(img_path):
            img_path = os.path.join(self.img_dir, f"{img_name}.jpeg")
            
        img = Image.open(img_path).convert('RGB')
        img_np = np.array(img)
        preprocessed = ben_graham_preprocessing(img_np, target_size=self.img_size)
        tensor = self.transform(preprocessed)
        label = int(row['diagnosis'])
        return tensor, label

## 6. Pretrained ResNet50 Model Architecture & Clinical Head

In [ ]:
import torch.nn as nn
from torchvision import models

class RetinalResNet50(nn.Module):
    def __init__(self, num_classes=5, pretrained=True, dropout_rate=0.3):
        super().__init__()
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        self.backbone = models.resnet50(weights=weights)
        in_features = self.backbone.fc.in_features
        
        self.backbone.fc = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate * 0.7),
            nn.Linear(512, num_classes)
        )
        
    def forward(self, x):
        return self.backbone(x)
        
    def get_cam_target_layer(self):
        return self.backbone.layer4[-1]

## 7. Model Training & Evaluation with Quadratic Weighted Kappa (QWK)

Diabetic retinopathy severity grading is ordinal (0 < 1 < 2 < 3 < 4).
**Quadratic Weighted Kappa (QWK)** penalizes distant classification errors (e.g. misclassifying Grade 4 as Grade 0) far more severely than adjacent errors (Grade 1 vs Grade 2).

In [ ]:
from sklearn.metrics import cohen_kappa_score, accuracy_score, confusion_matrix
import time

def train_epoch(model, dataloader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        if scaler is not None:
            with torch.amp.autocast(device_type='cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        
    return running_loss / len(dataloader.dataset), accuracy_score(all_labels, all_preds)

def evaluate_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    acc = accuracy_score(all_labels, all_preds)
    return running_loss / len(dataloader.dataset), acc, qwk, all_preds, all_labels

## 8. Explainable AI (Grad-CAM Implementation)

Generate visual gradient heatmaps on the final convolutional layer of ResNet50 to visualize the retinal lesions (microaneurysms, hemorrhages, exudates) responsible for the model's diagnostic grade.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        
        self.target_layer.register_forward_hook(lambda m, i, o: setattr(self, 'activations', o.detach()))
        self.target_layer.register_full_backward_hook(lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))
        
    def generate_heatmap(self, input_tensor, target_class=None):
        self.model.zero_grad()
        tensor = input_tensor.clone().detach().requires_grad_(True)
        logits = self.model(tensor)
        
        if target_class is None:
            target_class = int(torch.argmax(logits, dim=1).item())
            
        score = logits[0, target_class]
        score.backward()
        
        weights = torch.mean(self.gradients, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1, keepdim=True)
        cam = torch.relu(cam).squeeze().cpu().numpy()
        
        if np.max(cam) - np.min(cam) > 1e-8:
            cam = (cam - np.min(cam)) / (np.max(cam) - np.min(cam))
        else:
            cam = np.zeros_like(cam)
            
        h, w = input_tensor.shape[2], input_tensor.shape[3]
        return cv2.resize(cam, (w, h), interpolation=cv2.INTER_LINEAR), target_class

def overlay_heatmap(image_rgb, heatmap, alpha=0.5):
    heatmap_uint8 = np.uint8(255 * heatmap)
    colored_heatmap = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    colored_heatmap_rgb = cv2.cvtColor(colored_heatmap, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(image_rgb, 1.0 - alpha, colored_heatmap_rgb, alpha, 0)

## 9. Save Checkpoint for Streamlit Web Application
Download this `.pth` file to your local computer or GitHub repository for the Streamlit web deployment.

In [ ]:
# Save model weights
# torch.save(model.state_dict(), 'dr_resnet50_best.pth')
print("Model ready for clinical decision support deployment!")